<a href="https://colab.research.google.com/github/ezequielcabeja/popout-mcts-id3-ia/blob/main/%5Bfinal_project%5Dpopout_game_mcts_id3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Estratégias de Busca Adversarial e Árvores de Decisão Aplicadas ao Jogo PopOut**



**Projeto da Disciplina de Inteligência Artificial na Universidade do Porto**

*Professor: Francesco Renna*

*Alunos:Ezequiel Paulo,Flávia Figueiredo,Telma Freitas*

#**Objetivos do Projeto**





Este projeto tem como objetivo principal o design e a implementação de um agente inteligente capaz de jogar PopOut, integrando diferentes paradigmas de Inteligência Artificial para a tomada de decisão. O trabalho foca-se no desenvolvimento do algoritmo Monte Carlo Tree Search (MCTS), utilizando a métrica UCT, e na criação de um classificador baseado em Árvores de Decisão através do procedimento ID3. Um elemento central da metodologia consiste na utilização do MCTS para gerar um conjunto de dados sintético que, posteriormente, serve de base para o treino da árvore de decisão. Desta forma, procura-se avaliar a eficácia de modelos aprendidos indutivamente em comparação com métodos de procura adversarial, analisando o desempenho dos agentes em cenários de jogo contra humanos e entre diferentes algoritmos.

##**PopOut Game**






O jogo PopOut é uma variante do Connect-4. Ele começa da mesma forma que o jogo tradicional, com um tabuleiro vazio e os jogadores se revezando para colocar seus discos coloridos no tabuleiro. Durante cada turno, um jogador pode adicionar um disco do topo ou, se tiver discos da sua cor na fileira inferior,pode remover um disco. Ao remover um disco da parte inferior, todos os discos acima dele descem uma casa, alterando sua relação com o resto do tabuleiro e mudando as possibilidades de conexão. O primeiro jogador a conectar quatro de seus discos horizontalmente, verticalmente ou diagonalmente vence o jogo.

Neste projeto, vamos implementar este jogo em Python e, em seguida, implementar o algoritmo de Busca em Árvore Monte Carlo (MCTS) para jogá-lo. Depois, faremos uma implementação do algoritmo Id3, treinando-a com um conjunto de dados criado pelo algoritmo MCTS, para que possa-se jogar o PopOut com o Id3 também. Embora o algoritmo Id3 não seja normalmente usado para esse tipo de jogo, faremos este experimento neste projeto.




##Implementatação do Jogo PopOut




A seguir, apresentamos a implementação do jogo PopOut, levando em consideração as regras do jogo e também as especificações deste projeto, como a possibilidade de declarar empate após 3 tabuleiros completos.

**Bibliotecas utilizadas**

In [ ]:
import numpy as np

**Implementação do Jogo PopOut**

In [ ]:

# Size of the board
ROWS = 6
COLS = 7


class PopOutGame:
    def __init__(self, rows=ROWS, cols=COLS):
        self.rows = rows
        self.cols = cols

        # 0: vazio, 1: jogador 1 (X), 2: jogador 2 (O)
        self.board = np.zeros((rows, cols), dtype=int)

        self.current_player = 1 # 1 começa sempre

        # Histórico de estados (para regra de repetição)
        self.history = {}

        # Registar estado inicial
        self._register_state()

    # =========================================================
    # UTILITÁRIOS
    # =========================================================

    def opponent(self, player): # Retorna o oponente do jogador atual
        return 3 - player

    def copy(self): # Cria cópia profunda do jogo (ESSENCIAL para MCTS)
        new_game = PopOutGame(self.rows, self.cols)
        new_game.board = self.board.copy()
        new_game.current_player = self.current_player
        new_game.history = self.history.copy()  # Compartilha histórico para manter contagem de estados
        return new_game

    def _get_state_key(self): # Gera chave única para o estado atual (ESSENCIAL para MCTS)
        return tuple(map(tuple, self.board))

    def _register_state(self): # Registra o estado atual no histórico para contagem de repetições
        state = self._get_state_key()
        self.history[state] = self.history.get(state, 0) + 1

    def is_full(self): # Verifica se o tabuleiro está cheio (ESSENCIAL para regra de empate por tabuleiro cheio)
        return np.all(self.board != 0)

    def display(self): # Exibe o tabuleiro e informações do jogo
        symbols = {0: ".", 1: "X", 2: "O"}

        print("\n" + " ".join(map(str, range(self.cols))))
        for row in self.board:
            print(" ".join(symbols[cell] for cell in row))

        print(f"[bold magenta]Vez do jogador: {symbols[self.current_player]}[/bold magenta]")

    # =========================================================
    # MOVIMENTOS
    # =========================================================

    def get_valid_moves(self): # Retorna lista de movimentos válidos (ESSENCIAL para MCTS)
        moves = []

        # Drops
        for c in range(self.cols):
            if self.board[0][c] == 0:
                moves.append(('drop', c))

        # Pops
        for c in range(self.cols):
            if self.board[self.rows - 1][c] == self.current_player:
                moves.append(('pop', c))

        return moves

    def make_move(self, move_type, col): # Executa um movimento
        if (move_type, col) not in self.get_valid_moves():
            return False

        # DROP
        if move_type == 'drop':
            for r in reversed(range(self.rows)):
                if self.board[r][col] == 0:
                    self.board[r][col] = self.current_player
                    break

        # POP
        elif move_type == 'pop':
            for r in range(self.rows - 1, 0, -1):
                self.board[r][col] = self.board[r - 1][col]
            self.board[0][col] = 0

        # Registar novo estado
        self._register_state()

        # Trocar jogador
        self.current_player = self.opponent(self.current_player)

        return True

    def get_next_states(self): # Gera todos os estados possíveis a partir do estado atual (ESSENCIAL para MCTS)
        states = []

        for move in self.get_valid_moves():
            new_game = self.copy()
            new_game.make_move(move[0], move[1])
            states.append((move, new_game))

        return states

    # =========================================================
    # VERIFICAÇÃO DE VITÓRIA
    # =========================================================

    def check_winner(self, last_move_type=None): # Verifica se há um vencedor, considerando a regra especial do POP

        # Regra especial do POP
        if last_move_type == 'pop':
            if self._is_winning(self.opponent(self.current_player)):
                return self.opponent(self.current_player)

            if self._is_winning(self.current_player):
                return self.current_player

            return None

        # Verificação normal
        for p in [1, 2]:
            if self._is_winning(p):
                return p

        return None

    def _is_winning(self, player): # Verifica se o jogador tem 4 em linha

        # Horizontal
        for r in range(self.rows):
            for c in range(self.cols - 3):
                if all(self.board[r][c + i] == player for i in range(4)):
                    return True

        # Vertical
        for r in range(self.rows - 3):
            for c in range(self.cols):
                if all(self.board[r + i][c] == player for i in range(4)):
                    return True

        # Diagonais
        for r in range(self.rows - 3):
            for c in range(self.cols - 3):
                if all(self.board[r + i][c + i] == player for i in range(4)):
                    return True
                if all(self.board[r + 3 - i][c + i] == player for i in range(4)):
                    return True

        return False

    # =========================================================
    # EMPATES
    # =========================================================

    def check_draw(self): # Verifica condições de empate (repetição de estado e/ ou tabuleiro cheio)
        state = self._get_state_key()

        # Regra 3: repetição
        if self.history.get(state, 0) >= 3:
            return "draw_repetition"

        # Regra 2: tabuleiro cheio
        if self.is_full():
            pop_moves = [c for c in range(self.cols)
                         if self.board[self.rows - 1][c] == self.current_player]

            return "draw_full_board", pop_moves

        return None
class GameExit(Exception):
    #Exceção personalizada para sair do jogo de forma controlada
    pass


#**Algoritmos utilizados para implementar o PopOut**






##**Monte Carlo Tree Search**


O Monte Carlo Tree Search (MCTS) é um algoritmo de busca que utiliza simulações estatísticas para prever as melhores jogadas. Diferente de buscas exaustivas, ele foca nos caminhos mais promissores, equilibrando a exploração de novas jogadas e a análise de estratégias que já trouxeram vitórias. No PopOut, ele é utilizado para gerar um dataset especialista, simulando milhares de partidas para ensinar à Árvore de Decisão ID3 quais movimentos (como colocar ou remover peças) oferecem maior probabilidade de sucesso.

##Implementação do Algoritmo Monte Carlo Puro e Monte Carlo com Heurística específica para o jogo PopOut



In [ ]:
# mcts_ai.py
#from game import PopOutGame
import math
import random
import pickle
from datetime import datetime


class MCTSNode:
    """Nó da árvore MCTS"""

    def __init__(self, game_state, parent=None, move=None):
        self.game_state = game_state
        self.parent = parent
        self.move = move
        self.children = []
        self.visits = 0
        self.wins = 0
        self.untried_moves = game_state.get_valid_moves()

    def uct_value(self, exploration_constant=1.20):#testar valores diferentes!!
        if self.visits == 0:
            return float('inf')

        exploitation = self.wins / self.visits
        exploration = exploration_constant * math.sqrt(math.log(self.parent.visits) / self.visits)

        if self.game_state.current_player != self.parent.game_state.current_player:
            exploitation = 1 - exploitation

        return exploitation + exploration

    def expand(self):
        move = random.choice(self.untried_moves)
        self.untried_moves.remove(move)

        new_game = self.game_state.copy()
        new_game.make_move(move[0], move[1])

        child_node = MCTSNode(new_game, parent=self, move=move)
        self.children.append(child_node)
        return child_node

    def is_fully_expanded(self):
        return len(self.untried_moves) == 0

    def best_child(self, exploration_constant=1.20):
        return max(self.children, key=lambda c: c.uct_value(exploration_constant))

    def update(self, result):
        self.visits += 1
        self.wins += result


class MCTS:
    def __init__(self, iterations=700, exploration_constant=1.20):
        self.iterations = iterations
        self.exploration_constant = exploration_constant
        self.dataset = []  # (state, move) para decision tree

    def select(self, node):
        while not node.game_state.check_winner() and node.game_state.get_valid_moves(): #Enquanto o jogo não acabou E ainda existem movimentos possíveis:
            if not node.is_fully_expanded(): #Se o nó atual tem movimentos que ainda não foram explorados:
                return node.expand()
            else:
                node = node.best_child(self.exploration_constant)
        return node

    def simulate(self, game_state):
        sim_game = game_state.copy()
        starting_player = game_state.current_player
        move_count = 0
        max_moves = 50

        while move_count < max_moves:
            winner = sim_game.check_winner()
            draw = sim_game.check_draw()
            if winner is not None  or draw is not None:
                break

            moves = sim_game.get_valid_moves()
            if not moves:
                break

            #  MCTS puro: escolha completamente aleatória
            move = random.choice(moves)

            sim_game.make_move(move[0], move[1])
            move_count += 1

        winner = sim_game.check_winner()


        # 🤝 empate neutro (melhor que 0 puro)
        if winner is None:
            return 0.5

        return 1 if winner == starting_player else 0

    def backpropagate(self, node, result):
        while node is not None:
            node.update(result)
            result = 1 - result
            node = node.parent

    def get_best_move(self, game_state, return_confidence=False):
        root = MCTSNode(game_state)

        for _ in range(self.iterations):
            node = self.select(root)
            result = self.simulate(node.game_state)
            self.backpropagate(node, result)

        if not root.children:
            moves = game_state.get_valid_moves()
            if moves:
                best_move = moves[0]
                confidence = 0
            else:
                return None
        else:
            best_child = max(root.children, key=lambda c: c.visits)
            best_move = best_child.move
            confidence = best_child.wins / best_child.visits if best_child.visits > 0 else 0

        # Salvar no dataset
        self._add_to_dataset(game_state, best_move)

        if return_confidence:
            return best_move, confidence
        return best_move

    def _add_to_dataset(self, game_state, move):
        """Adiciona par (estado, movimento) ao dataset"""
        state_key = {
            'board': game_state.board.tolist(),
            'current_player': game_state.current_player
        }

        self.dataset.append({
            'state': state_key,
            'move': move
        })

    def save_dataset(self, filename=None):
        if filename is None:
            filename = f"mcts_dataset_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"

        with open(filename, 'wb') as f:
            pickle.dump(self.dataset, f)

        print(f"✅ Dataset salvo em {filename} com {len(self.dataset)} exemplos")
        return filename


class MCTS_Heuristic(MCTS):
    """
    MCTS com heurística OTIMIZADA

    """

    def _count_pieces_in_direction(self, board, row, col, player, dr, dc):
        """Conta peças consecutivas em uma direção"""
        count = 0
        r, c = row + dr, col + dc
        rows, cols = len(board), len(board[0])

        while 0 <= r < rows and 0 <= c < cols and board[r][c] == player:
            count += 1
            r += dr
            c += dc

        return count

    def _evaluate_move_fast(self, game_state, move, player):
        """
        Avalia movimento rapidamente (sem cópia!)
        """
        board = game_state.board
        col = move[1]

        # Encontrar linha onde a peça cairia
        row = -1
        for r in range(game_state.rows - 1, -1, -1):
            if board[r][col] == 0:
                row = r
                break

        if row == -1:
            return -1000  # Movimento inválido

        # 1. VITÓRIA IMEDIATA?
        directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
        for dr, dc in directions:
            count = 1
            count += self._count_pieces_in_direction(board, row, col, player, dr, dc)
            count += self._count_pieces_in_direction(board, row, col, player, -dr, -dc)

            if count >= 4:
                return 10000  # Vitória!

        # 2. BLOQUEIO? (verificar se oponente venceria)
        opponent = 3 - player
        for dr, dc in directions:
            count = 1
            count += self._count_pieces_in_direction(board, row, col, opponent, dr, dc)
            count += self._count_pieces_in_direction(board, row, col, opponent, -dr, -dc)

            if count >= 3:  # Se oponente tem 3, precisa bloquear
                return 9000

        # 3. Avaliar ameaças (3 em linha, 2 em linha)
        score = 0
        for dr, dc in directions:
            count = 1
            count += self._count_pieces_in_direction(board, row, col, player, dr, dc)
            count += self._count_pieces_in_direction(board, row, col, player, -dr, -dc)

            if count == 3:
                score += 100
            elif count == 2:
                score += 10
            elif count == 1:
                score += 1

        # 4. Bônus por posição (centro é melhor)
        if col == 3:
            score += 5
        elif col in [2, 4]:
            score += 3
        elif col in [1, 5]:
            score += 1

        # 5. Bônus por altura (quanto mais baixo, melhor)
        score += (game_state.rows - row) * 2

        return score

    def simulate(self, game_state):
        sim_game = game_state.copy()
        starting_player = game_state.current_player
        move_count = 0
        max_moves = 50

        while move_count < max_moves:
            winner = sim_game.check_winner()
            draw = sim_game.check_draw()
            if winner is not None or draw is not None:
                break

            moves = sim_game.get_valid_moves()
            if not moves:
                break

            scored_moves = []
            for move in moves:
                score = self._evaluate_move_fast(sim_game, move, sim_game.current_player)
                scored_moves.append((score, move))

            best_move = max(scored_moves, key=lambda x: x[0])[1]

            if random.random() < 0.6:
                move = best_move
            else:
                move = random.choice(scored_moves[:3])[1]

            sim_game.make_move(move[0], move[1])
            move_count += 1

        winner = sim_game.check_winner()

        if winner is None:
            return 0.5

        return 1 if winner == starting_player else 0

class PopOutAI:
    """Wrapper para usar MCTS no jogo"""

    def __init__(self, algorithm='mcts', iterations=700):
        if algorithm == 'mcts':
            self.mcts = MCTS(iterations=iterations)
        elif algorithm == 'mcts_heuristic':
            self.mcts = MCTS_Heuristic(iterations=iterations)
        else:
            self.mcts = MCTS(iterations=iterations)

        self.algorithm_name = algorithm

    def get_move(self, game_state, return_confidence=False):
        return self.mcts.get_best_move(game_state, return_confidence)

    def save_dataset(self, filename=None):
        return self.mcts.save_dataset(filename)

    def get_dataset(self):
        return self.mcts.dataset

##**Análises realizadas para decisões sobre os parâmetros do Monte Carlo**

Antes de tomar a decisão sobre os parâmetros que utilizaríamos para o código de implementação do Monte Carlo, realizamos alguns testes, levando em conta o trade-off entre a qualidade do código e a quantidade de tempo que precisaríamos para gerar os datasets para treinar o Id3.
Escolhemos realizar apenas 50 jogos para os testes e o Algoritmo inicialmente com 500 iterações, devido ao desafio do tempo

In [ ]:
def benchmark(config1, config2, num_jogos=50):
    vitorias_p1 = 0
    vitorias_p2 = 0
    empates = 0

    print(f"Iniciando: {config1['name']} vs {config2['name']}")

    for i in range(num_jogos):
        game = PopOutGame()
        # Alternar quem começa
        p1_vez = (i % 2 == 0)

        while True:
            # Seleciona qual IA deve jogar
            if game.current_player == 1:
                config = config1 if p1_vez else config2
            else:
                config = config2 if p1_vez else config1

            # Instancia a IA com as configurações do teste
            if config['tipo'] == 'heuristico':
                ia = MCTS_Heuristic(iterations=config['iters'], exploration_constant=config['c'])
            else:
                ia = MCTS(iterations=config['iters'], exploration_constant=config['c'])

            move = ia.get_best_move(game)
            game.make_move(move[0], move[1])

            winner = game.check_winner(last_move_type=move[0])
            if winner:
                if (winner == 1 and p1_vez) or (winner == 2 and not p1_vez):
                    vitorias_p1 += 1
                else:
                    vitorias_p2 += 1
                break

            if game.check_draw():
                empates += 1
                break

        if (i+1) % 10 == 0:
            print(f"Jogo {i+1} finalizado...")

    print("\n--- RESULTADOS ---")
    print(f"{config1['name']}: {vitorias_p1} vitórias")
    print(f"{config2['name']}: {vitorias_p2} vitórias")
    print(f"Empates: {empates}")

##Monte Carlo sem heurística x Monte Carlo com heurística

Implementamos um código Monte Carlo padrão, e um segundo com heurísticas que davam "conselhos" a serem seguidos para o algoritmo em algumas situações específicas, tais como momentos em que uma jogada em especial poderia definir o resultado final do jogo.

In [ ]:
p1 = {'name': 'MCTS Puro', 'tipo': 'puro', 'iters': 500, 'c': 1.41}
p2 = {'name': 'MCTS Heuristico', 'tipo': 'heuristico', 'iters': 500, 'c': 1.41}
#demora bastante para rodar, apenas descomentar se realmente quiser rodar o teste novamente
#benchmark(p1, p2, num_jogos=50)



*Rodando o código acima, obtivemos os seguintes resultados:*

*MCTS Puro: 17 vitórias*

*MCTS Heuristico: 33 vitórias*

*Empates: 0*


**Como o Monte Carlo com heurística demonstrou ter melhores resultados, ele foi o escolhido para a geração do dataset, visando a obtenção de uma base de dados com os melhores resultados possíveis para o treinamento da nossa Id3.**



##Constante de exploração do UCT padrão x  com menor valor

A constante de exploração define o quanto o nosso algoritmo estará predisposto a explorar novas possibilidades, ao invés de focar onde já está a ver resultados.
Testamos a hipótese de um algoritmo que explora um pouco menos.

In [ ]:
p1 = {'name': 'MCTS Heurístico 1.41', 'tipo': 'heuristico', 'iters': 500, 'c': 1.41}
p2 = {'name': 'MCTS Heurístico 1.00', 'tipo': 'heuristico', 'iters': 500, 'c': 1.20}
#demora bastante para rodar, apenas descomentar se realmente quiser rodar o teste novamente
#benchmark(p1, p2, num_jogos=50)

*Rodando o código acima, obtivemos os seguintes resultados:*

*MCTS Heurístico 1.41: 21 vitórias*

*MCTS Heurístico 1.20: 29 vitórias*

*Empates: 0*

**Notamos uma melhoria interessante nos resultados ao diminuir o valor da constante C. Portanto, escolhemos diminuir o valor da constance C para a geração do nosso dataset.**

##Número de Iterações

Realizamos testes do Monte Carlo com heuristica com 1000x500 iterações, 1000x600 iterações e 1000x700 iterações.
O algoritmo com 1000 iterações demonstrou resultados muito superiores às outras possibilidades. No entanto, devido ao desafio do tempo para gerar os datasets, acabamos optando por um número menor de iterações.

In [ ]:
p1 = {'name': 'MCTS 700 iters', 'tipo': 'heuristico', 'iters': 700, 'c': 1.20}
p2 = {'name': 'MCTS 1000 iters', 'tipo': 'heuristico', 'iters': 1000, 'c': 1.20}
#demora bastante para rodar, apenas descomentar se realmente quiser rodar o teste novamente

#benchmark(p1, p2, num_jogos=50)

*Resultados obtidos:*

*MCTS 700 iters: 16 vitórias*

*MCTS 1000 iters: 34 vitórias*

*Empates: 0*

Os resultados de 500 e 600 iterações x 1000 foram 15x 35 vitórias para o algoritmo com 1000 iterações. Portanto as 700 iterações apresentaram uma pequena vantagem em relação as outras opções, e foi a escolhida.

**Apesar de não apresentar o resultado ideal, 700 foi escolhido como o número de iterações viável para o andamento deste trabalho.**

##**Geração de DataSets**

Utilizamos o código abaixo para gerar o dataset necessário para treinar o algoritmo id3. Além dos tabuleiros com as jogadas realizadas ao longo da partida, neste dataset indicamos também quem foi o ganhador de cada partida, visando na criação de um dataset com mais informações relevantes para o treinamento do Id3.



In [ ]:

def generate_dataset_with_winner(target_examples=15000, iterations_per_move=700):
    ai_heur1 = PopOutAI(algorithm='mcts_heuristic', iterations=iterations_per_move)
    ai_heur = PopOutAI(algorithm='mcts_heuristic', iterations=iterations_per_move) #como o código com heurística demonstrou ser melhor, vamos jogar com heurística x com heurística, para garantir que ele tenha sempre as melhores jogadas

    dataset = []
    game_num = 0

    while len(dataset) < target_examples:
        game_num += 1
        game = PopOutGame()
        move_count = 0
        max_moves = 50
        game_history = []

        while move_count < max_moves:
            moves = game.get_valid_moves()
            if not moves:
                break

            if game.current_player == 1:
                move = ai_heur1.get_move(game)
            else:
                move = ai_heur.get_move(game)

            if move:
                game_history.append({
                    "state": {
                        "board": game.board.tolist(),
                        "current_player": game.current_player
                    },
                    "move": move
                })
                game.make_move(move[0], move[1])

            winner = game.check_winner()
            if winner:
                break

            draw_status = game.check_draw()
            if draw_status:
                if draw_status[0] != "draw_full_board":
                    break
            move_count += 1

        winner = game.check_winner()
        if winner is None:
            winner = 0

        for item in game_history:
            item["winner"] = 1 if item["state"]["current_player"] == winner else 0
            dataset.append(item)

        if game_num % 10 == 0:
            print(f"📊 Jogos: {game_num} | Exemplos: {len(dataset)}/{target_examples}")

    # --- PARTE PARA SALVAR NO DRIVE ---
    # Define o caminho dentro da pasta que você criou
    filename = f"/content/drive/MyDrive/PopOutDataset/dataset_winner_{target_examples}ex.pkl"

    with open(filename, 'wb') as f:
        pickle.dump(dataset, f)

    print(f"\n✔ Sucesso! Dataset salvo em: {filename}")
    print(f"✔ Total de exemplos coletados: {len(dataset)}")
    # ----------------------------------

    return dataset


**Definição do Tamanho do DataSet**


Para decidir o tamanho do dataset que geraria os melhores resultados, fizemos alguns testes práticos variando a quantidade de tabuleiros e a "inteligência" do algoritmo MCTS. Testamos bases com 5.000, 10.000 e 15.000 tabuleiros, todos rodando com 700 iterações do MCTS.

Também testamos uma hipótese: um dataset maior, mesmo que as jogadas fossem calculadas mais rapidamente, traria um resultado melhor?  Para isso, geramos uma base de 25.000 tabuleiros com apenas 100 iterações.

No fim, a configuração de 15.000 tabuleiros com 700 iterações foi a que apresentou os melhores dados para o treinamento. Percebemos que vale mais a pena ter exemplos de alta qualidade (com mais processamento por jogada) do que um volume gigante de jogadas menos precisas.

Tais testes foram realizados na própria implementação do Id3 para o PopOut, através da medida da Acurácia.



In [ ]:
####DEMORA MUITAS HORAS PARA SER EXECUTADO, PORTANTO EXECUTAR APENAS SE PRECISAR GERAR O DATASET#####
#meu_dataset = generate_dataset_with_winner(target_examples=15000, iterations_per_move=700)

##**Id3**





O ID3 (Iterative Dichotomiser 3) é um algoritmo  de aprendizado de máquina que constrói árvores de decisão baseando-se nos conceitos de Entropia e Ganho de Informação. Embora não seja o método padrão para jogos de tabuleiro complexos , realizaremos este experimento para testar sua eficácia na destilação de conhecimento. A ideia foi utilizar o ID3 para processar milhares de jogadas "especialistas" geradas pelo MCTS e transformá-las em um conjunto de regras lógicas e rápidas. Esse experimento nos permitiu avaliar como um modelo de classificação tradicional se comporta ao tentar mapear padrões estratégicos do PopOut, buscando uma inteligência que decida de forma instantânea sem a necessidade de cálculos pesados a cada turno.

#Algoritmo Id3 treinado com Íris DataSet

A seguir, implementamos o algoritmo Id3, sem a utilização de bibliotecas padrão para definir e treinar a árvore, ou seja, foi uma implementação mais completa.
Como o Id3 não é tipicamente  utilizado para este tipo de problemas de jogos de tabuleiro, antes de implementar o algoritmo com o jogo, o implementamos utilizando o clássico dataset Íris, dessa forma conseguimos ter uma visão mais realista da acurácia do algoritmo criado.

In [ ]:
import pandas as pd
import numpy as np
import math
from collections import Counter

In [ ]:


# =========================================================
# 1. CARREGAMENTO E PRÉ-PROCESSAMENTO
# =========================================================

df = pd.read_csv("iris.csv")

# Remover ID (se existir)
if 'ID' in df.columns:
    df = df.drop(columns=['ID'])

# Garantir que atributos são numéricos
for col in df.columns[:-1]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Shuffle + split manual (80/20)
df_shuffle = df.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(df_shuffle) * 0.8)

train_df = df_shuffle.iloc[:split_idx]
test_df = df_shuffle.iloc[split_idx:]


# =========================================================
# 2. ID3 IMPLEMENTAÇÃO
# =========================================================

def entropy(y):
    total = len(y)
    if total == 0:
        return 0

    counts = Counter(y)
    ent = 0

    for c in counts.values():
        p = c / total
        ent -= p * math.log2(p)

    return ent


def information_gain(df, attribute, threshold, target="class"):
    total_entropy = entropy(df[target])

    left = df[df[attribute] <= threshold]
    right = df[df[attribute] > threshold]

    if len(left) == 0 or len(right) == 0:
        return 0

    p_left = len(left) / len(df)
    p_right = len(right) / len(df)

    weighted_entropy = (
        p_left * entropy(left[target]) +
        p_right * entropy(right[target])
    )

    return total_entropy - weighted_entropy


def id3_manual(df, attributes, target="class", depth=0, max_depth=4):
    # Caso 1: todos iguais
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    # Caso 2: parar
    if depth >= max_depth or not attributes:
        return df[target].mode()[0]

    best_gain = -1
    best_attr = None
    best_threshold = None

    for attr in attributes:
        values = sorted(df[attr].unique())

        thresholds = [
            (values[i] + values[i + 1]) / 2
            for i in range(len(values) - 1)
        ]

        for t in thresholds:
            gain = information_gain(df, attr, t, target)

            if gain > best_gain:
                best_gain = gain
                best_attr = attr
                best_threshold = t

    if best_gain <= 0:
        return df[target].mode()[0]

    node_key = (best_attr, best_threshold)
    tree = {node_key: {}}

    left_df = df[df[best_attr] <= best_threshold]
    right_df = df[df[best_attr] > best_threshold]

    tree[node_key][f"<= {best_threshold:.2f}"] = id3_manual(
        left_df, attributes, target, depth + 1
    )
    tree[node_key][f"> {best_threshold:.2f}"] = id3_manual(
        right_df, attributes, target, depth + 1
    )

    return tree


# =========================================================
# 3. PREDIÇÃO E VISUALIZAÇÃO
# =========================================================

def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    (attr, threshold), branches = next(iter(tree.items()))

    val = sample[attr]

    if val <= threshold:
        return predict(branches[f"<= {threshold:.2f}"], sample)
    else:
        return predict(branches[f"> {threshold:.2f}"], sample)


def print_tree(tree, indent="", is_last=True):
    if not isinstance(tree, dict):
        prefix = "└── " if is_last else "├── "
        print(f"{indent}{prefix}CLASSE: {tree}")
        return

    (attr, threshold), branches = next(iter(tree.items()))
    items = list(branches.items())

    for i, (condition, subtree) in enumerate(items):
        is_last_child = (i == len(items) - 1)
        prefix = "└── " if is_last_child else "├── "

        # Melhorar visual
        condition_clean = condition.replace("<=", "≤").replace(">", ">")

        print(f"{indent}{prefix}[{attr} {condition_clean}]")

        new_indent = indent + ("    " if is_last_child else "│   ")

        print_tree(subtree, new_indent, is_last_child)


# =========================================================
# 4. EXECUÇÃO
# =========================================================

print("\nIniciando Treinamento da Árvore ID3...\n")

features = [col for col in df.columns if col != "class"]

tree = id3_manual(train_df, features)

# ----------------------------
# ÁRVORE
# ----------------------------
print("=== ÁRVORE DE DECISÃO (ID3) ===\n")
print_tree(tree)

# ----------------------------
# AVALIAÇÃO
# ----------------------------
corretos = 0

for _, row in test_df.iterrows():
    if predict(tree, row) == row["class"]:
        corretos += 1

accuracy = (corretos / len(test_df)) * 100

print("\n=== RESULTADOS DO MODELO ID3 ===\n")
print(f"{'Total de exemplos:':25} {len(test_df)}")
print(f"{'Corretos:':25} {corretos}")
print(f"{'Incorretos:':25} {len(test_df) - corretos}")

print(f"\nAcurácia no Conjunto de Teste: {accuracy:.2f}%")

# ----------------------------
# EXEMPLO
# ----------------------------
exemplo_teste = {
    "sepallength": 5.1,
    "sepalwidth": 3.5,
    "petallength": 1.4,
    "petalwidth": 0.2
}

resultado = predict(tree, exemplo_teste)

print("\n=== EXEMPLO DE TESTE ===\n")

for k, v in exemplo_teste.items():
    print(f"{k:15}: {v}")

print(f"\n{'Predição:':15} {resultado}")


Iniciando Treinamento da Árvore ID3...

=== ÁRVORE DE DECISÃO (ID3) ===

├── [petallength ≤ 2.45]
│   ├── CLASSE: Iris-setosa
└── [petallength > 2.45]
    ├── [petalwidth ≤ 1.75]
    │   ├── [petallength ≤ 4.95]
    │   │   ├── CLASSE: Iris-versicolor
    │   └── [petallength > 4.95]
    │       ├── [petalwidth ≤ 1.55]
    │       │   ├── CLASSE: Iris-virginica
    │       └── [petalwidth > 1.55]
    │           └── CLASSE: Iris-versicolor
    └── [petalwidth > 1.75]
        ├── [petallength ≤ 4.85]
        │   ├── [sepallength ≤ 5.95]
        │   │   ├── CLASSE: Iris-versicolor
        │   └── [sepallength > 5.95]
        │       └── CLASSE: Iris-virginica
        └── [petallength > 4.85]
            └── CLASSE: Iris-virginica

=== RESULTADOS DO MODELO ID3 ===

Total de exemplos:        30
Corretos:                 28
Incorretos:               2

Acurácia no Conjunto de Teste: 93.33%

=== EXEMPLO DE TESTE ===

sepallength    : 5.1
sepalwidth     : 3.5
petallength    : 1.4
petalwidth 

#Algoritmo Id3 para o PopOut

Conforme já mencionado anteriormente, para o treinamento deste algoritmo para o jogo PopOut, utilizamos uma base gerada a partir do algoritmo Monte Carlo com heurística, configurado para 700 iterações por jogada e com constante de exploração igual a  1.20. O tamanho do dataset escolhido foio de 15000 exemplos de tabuleiros.

In [ ]:
import pandas as pd
import numpy as np
import math
from collections import Counter

import pickle

In [ ]:

# =========================================================
# 1. CARREGAMENTO
# =========================================================

df = pd.read_csv("dataset_winner_15000ex.csv")
# =========================================================
# CRIAR LABEL
# =========================================================

df["label"] = (
    df["move_type"].astype(str)
    + "_"
    + df["move_col"].astype(str)
)

# separar treino/teste
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]


# =========================================================
# 2. ID3
# =========================================================

def entropy(y):
    total = len(y)
    counts = Counter(y)

    ent = 0
    for c in counts.values():
        p = c / total
        ent -= p * math.log2(p)

    return ent


def information_gain(df, attribute, threshold, target="label"):
    total_entropy = entropy(df[target])

    left = df[df[attribute] <= threshold]
    right = df[df[attribute] > threshold]

    if len(left) == 0 or len(right) == 0:
        return 0

    p_left = len(left) / len(df)
    p_right = len(right) / len(df)

    return total_entropy - (
        p_left * entropy(left[target]) +
        p_right * entropy(right[target])
    )


def id3_manual(df, attributes, target="label", depth=0, max_depth=10):

    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    if depth >= max_depth or not attributes:
        return df[target].mode()[0]

    best_gain = -1
    best_attr = None
    best_threshold = None

    for attr in attributes:
        values = sorted(df[attr].unique())

        thresholds = [
            (values[i] + values[i+1]) / 2
            for i in range(len(values)-1)
        ]

        for t in thresholds:
            gain = information_gain(df, attr, t, target)

            if gain > best_gain:
                best_gain = gain
                best_attr = attr
                best_threshold = t

    if best_gain <= 0:
        return df[target].mode()[0]

    node_key = (best_attr, best_threshold)
    tree = {node_key: {}}

    left_df = df[df[best_attr] <= best_threshold]
    right_df = df[df[best_attr] > best_threshold]

    tree[node_key][f"<= {best_threshold:.2f}"] = id3_manual(
        left_df, attributes, target, depth + 1
    )

    tree[node_key][f"> {best_threshold:.2f}"] = id3_manual(
        right_df, attributes, target, depth + 1
    )

    return tree


# =========================================================
# 3. PREDIÇÃO
# =========================================================

def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    (attr, threshold), branches = next(iter(tree.items()))

    val = sample[attr]

    if val <= threshold:
        return predict(branches[f"<= {threshold:.2f}"], sample)
    else:
        return predict(branches[f"> {threshold:.2f}"], sample)


def board_to_dict(board, current_player=1):
    sample = {}

    rows, cols = board.shape

    for r in range(rows):
        for c in range(cols):
            sample[f"cell_{r}_{c}"] = board[r][c]

    sample["current_player"] = current_player

    return sample

# =========================================================
# 4. VISUALIZAÇÃO
# =========================================================

def print_tree(tree, indent="", is_last=True):
    if not isinstance(tree, dict):
        prefix = "└── " if is_last else "├── "
        print(f"{indent}{prefix}CLASSE: {tree}")
        return

    (attr, threshold), branches = next(iter(tree.items()))
    items = list(branches.items())

    for i, (condition, subtree) in enumerate(items):
        is_last_child = (i == len(items) - 1)
        prefix = "└── " if is_last_child else "├── "

        condition_clean = condition.replace("<=", "≤")

        print(f"{indent}{prefix}[{attr} {condition_clean}]")

        new_indent = indent + ("    " if is_last_child else "│   ")

        print_tree(subtree, new_indent, is_last_child)


def train():
    df = pd.read_csv("dataset_winner_15000ex.csv")

    df["label"] = (
        df["move_type"].astype(str)
        + "_"
        + df["move_col"].astype(str)
    )

    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx]

    features = [
        col for col in df.columns
        if col not in ["label", "move_type", "move_col", "winner"]
    ]

    tree = id3_manual(train_df, features)

    return tree


def save_model(tree, path="id3_model.pkl"):
    with open(path, "wb") as f:
        pickle.dump(tree, f)

def load_model(path="id3_model.pkl"):
    with open(path, "rb") as f:
        return pickle.load(f)

if __name__ == "__main__":

    print("\nIniciando Treinamento ID3 (PopOut)...\n")

    df = pd.read_csv("dataset_winner_15000ex.csv")
    df["label"] = (
        df["move_type"].astype(str)
        + "_"
        + df["move_col"].astype(str)
    )
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]

    features = [
            col for col in df.columns
            if col not in ["label", "move_type", "move_col", "winner"]
        ]

    tree = id3_manual(train_df, features)

    print("=== ÁRVORE DE DECISÃO (PopOut) ===\n")
    print_tree(tree)

    # =============================
    # AVALIAÇÃO
    # =============================
    corretos = 0

    for _, row in test_df.iterrows():
        sample = row.to_dict()
        if predict(tree, sample) == row["label"]:
            corretos += 1

    accuracy = (corretos / len(test_df)) * 100

    print("\n=== RESULTADOS DO MODELO ID3 ===\n")
    print(f"{'Total de exemplos:':25} {len(test_df)}")
    print(f"{'Corretos:':25} {corretos}")
    print(f"{'Incorretos:':25} {len(test_df) - corretos}")

    print(f"\nAcurácia no Conjunto de Teste: {accuracy:.2f}%")

    # =============================
    # TESTE
    # =============================
    print("\n=== TESTE COM TABULEIRO REAL ===\n")

    example_board = np.zeros((6, 7))
    sample = board_to_dict(example_board)

    prediction = predict(tree, sample)

    print(f"{'Predição:':15} {prediction}")

    # =============================
    # GUARDAR MODELO (MUITO IMPORTANTE)
    # =============================
    save_model(tree)
    print("\n[✔] Modelo guardado como id3_model.pkl")



Iniciando Treinamento ID3 (PopOut)...

=== ÁRVORE DE DECISÃO (PopOut) ===

├── [cell_2_3 ≤ 0.50]
│   ├── [cell_2_2 ≤ 0.50]
│   │   ├── [cell_5_2 ≤ 0.50]
│   │   │   ├── [cell_5_1 ≤ 0.50]
│   │   │   │   ├── [cell_4_3 ≤ 0.50]
│   │   │   │   │   ├── [cell_5_5 ≤ 1.50]
│   │   │   │   │   │   ├── [current_player ≤ 1.50]
│   │   │   │   │   │   │   ├── [cell_5_3 ≤ 0.50]
│   │   │   │   │   │   │   │   ├── [cell_5_5 ≤ 0.50]
│   │   │   │   │   │   │   │   │   ├── [cell_5_4 ≤ 0.50]
│   │   │   │   │   │   │   │   │   │   ├── CLASSE: drop_3
│   │   │   │   │   │   │   │   │   └── [cell_5_4 > 0.50]
│   │   │   │   │   │   │   │   │       └── CLASSE: drop_3
│   │   │   │   │   │   │   │   └── [cell_5_5 > 0.50]
│   │   │   │   │   │   │   │       ├── [cell_0_5 ≤ 0.50]
│   │   │   │   │   │   │   │       │   ├── CLASSE: drop_4
│   │   │   │   │   │   │   │       └── [cell_0_5 > 0.50]
│   │   │   │   │   │   │   │           └── CLASSE: drop_3
│   │   │   │   │   │   │   └── [cell_5_3 > 0.50]
│   

**Acurácia do Id3**

No que diz respeito à performance do algoritmo ID3, observou-se que o modelo treinado com a base de 15.000 exemplos e 700 iterações (geradas via simulação de Monte Carlo) obteve um desempenho superior às demais configurações. A acurácia atingiu 34,42%, consolidando-se como o resultado mais expressivo entre os cenários testados para o domínio do jogo de tabuleiro.

#**Interface do Jogo PopOut**




In [ ]:
#from shutil import move

from rich.console import Console
from rich.table import Table
from rich.prompt import Prompt
import time
import random
#import numpy as np
#from mcts import MCTS, MCTS_Heuristic
#from utils import GameExit
#from game import PopOutGame
#from id3_popout_dataset import predict, load_model




In [ ]:

console = Console()


class GameInterface:
    def __init__(self):
        self.game = PopOutGame()
        self.mcts = MCTS(iterations=1000)
        self.mcts_heuristic = MCTS_Heuristic(iterations=500)

        console.print("\n[bold cyan]A carregar modelo ID3...[/bold cyan]")

        try:
            self.id3_model = load_model()
            console.print("[bold green] ID3 ativo![/bold green]")
        except:
            console.print("[bold yellow]ID3 indisponível ... fallback para Random[/bold yellow]")
            self.id3_model = None

        self.ai_strategies = {
            "MCTS": self.mcts_ai,
            "MCTS_Heuristico": self.mcts_heuristic_ai,
            "Random": self.random_ai,
            "ID3": self.id3_ai
        }

        self.player_ai = {}
        self.player_types = {}

       # self.ai_strategies["NovaIA"] = self.nova_funcao

    def board_to_dict(self):

        sample = {}

        rows, cols = self.game.board.shape

        for r in range(rows):
            for c in range(cols):
                sample[f"cell_{r}_{c}"] = self.game.board[r][c]

        sample["current_player"] = self.game.current_player

        return sample


    # =========================================================
    # VISUALIZAÇÃO
    # =========================================================

    def render_board(self):
        table = Table(show_header=True, header_style="bold blue")

        # Cabeçalhos (colunas)
        for col in range(self.game.cols):
            table.add_column(str(col), justify="center")

        symbols = {0: "·", 1: "[red]X[/red]", 2: "[yellow]O[/yellow]"}

        for row in self.game.board:
            table.add_row(*[symbols[cell] for cell in row])

        console.print(table)

        current = self.game.current_player
        name = self.player_names[current]

        console.print(f"\n[bold magenta]Turno: {name} ({symbols[current]})[/bold magenta]")

    # =========================================================
    # INPUT DO UTILIZADOR
    # =========================================================

    def get_player_move(self):
        valid_moves = self.game.get_valid_moves()

        console.print(f"\nMovimentos disponíveis: {valid_moves}")
        console.print("[dim]Digite 'q' ou 'exit' para sair[/dim]")

        while True:
            # NÃO usar "choices" aqui (permite q/exit)
            move_type = Prompt.ask("Tipo de jogada").strip().lower()

            # VERIFICAÇÃO DE SAÍDA (AQUI!)
            if move_type in ["q", "exit"]:
                confirm = Prompt.ask("Tem certeza que quer sair? (y/n)", choices=["y", "n"])
                if confirm == "y":
                    raise GameExit()
                else:
                    continue

            # agora pede coluna
            col = Prompt.ask("Coluna").strip().lower()

            # VERIFICAÇÃO DE SAÍDA (TAMBÉM AQUI!)
            if col in ["q", "exit"]:
                confirm = Prompt.ask("Tem certeza que quer sair? (y/n)", choices=["y", "n"])
                if confirm == "y":
                    raise GameExit()
                else:
                    continue

            # valida número
            try:
                col = int(col)
            except:
                console.print("[red]Coluna inválida[/red]")
                continue

            # valida tipo de jogada
            if move_type not in ["drop", "pop"]:
                console.print("[red]Tipo inválido (use drop/pop)[/red]")
                continue

            # valida jogada completa
            if (move_type, col) in valid_moves:
                return move_type, col
            else:
                console.print("[red]Jogada inválida. Tente novamente.[/red]")

    # =========================================================
    # MODOS DE JOGO
    # =========================================================
    def mcts_ai(self):
        return self.mcts.get_best_move(self.game)

    def mcts_heuristic_ai(self):
        return self.mcts_heuristic.get_best_move(self.game)

    def choose_ai(self, player_label):
        console.print(f"\n[bold cyan]Escolha IA {player_label}[/bold cyan]")
        console.print("1 - MCTS")
        console.print("2 - MCTS (Heurístico)")
        console.print("3 - Aleatório")
        console.print("4 - ID3 (Aprendido)")

        choice = Prompt.ask("Opção", choices=["1", "2", "3", "4"])

        mapping = {
            "1": "MCTS",
            "2": "MCTS_Heuristico",
            "3": "Random",
            "4": "ID3"
            }

        return mapping[choice]

    def id3_ai(self):
        if self.id3_model is None:
            return self.random_ai()

        try:
            sample = self.board_to_dict()
            prediction = predict(self.id3_model, sample)

            move_type, col = prediction.split("_")
            col = int(col)
            move = (move_type, col)

            if move in self.game.get_valid_moves():
                return move
            else:
                console.print(f"[yellow]ID3 sugeriu inválido: {move} fallback[/yellow]")
                return self.random_ai()

        except Exception as e:
            console.print(f"[red]Erro ID3: {e}[/red]")
            return self.random_ai()

    def choose_mode(self):
        console.print("\n[bold cyan]Escolha o modo de jogo:[/bold cyan]")
        console.print("1 - Humano_A vs Humano_B")
        console.print("2 - Humano_A vs IA")
        console.print("3 - IA vs IA")

        choice = Prompt.ask("Opção", choices=["1", "2", "3"])
        return int(choice)

    def get_move(self):
        player = self.game.current_player

        if self.player_types[player] == "human":
            return self.get_player_move()

        elif self.player_types[player] == "ai":
            ai_type = self.player_ai[player]

            #TESTE DE TEMPO (OPCIONAL)
            time.sleep(0.5) # pequena pausa para melhor experiência visual

            strategy = self.ai_strategies[ai_type]

            console.print(f"[cyan]{ai_type} está a jogar...[/cyan]")
            return strategy()


    # =========================================================
    # LOOP PRINCIPAL
    # =========================================================


    def run(self):
        moves_count = 0 # CONTADOR DE JOGADAS PARA TESTES DE TEMPO (OPCIONAL)
        try:
            mode = self.choose_mode()

            # Reset jogo
            self.game = PopOutGame()

            if mode == 1:
                self.player_types = {1: "human", 2: "human"}
                self.player_names = {1: "Humano_A", 2: "Humano_B"}

            elif mode == 2:
                ai_type = self.choose_ai("Jogador 2")

                self.player_types = {1: "human", 2: "ai"}
                self.player_names = {1: "Humano_A", 2: ai_type}
                self.player_ai = {2: ai_type}

            elif mode == 3:
                ai1 = self.choose_ai("Jogador 1")
                ai2 = self.choose_ai("Jogador 2")

                self.player_types = {1: "ai", 2: "ai"}
                self.player_names = {1: ai1, 2: ai2}
                self.player_ai = {1: ai1, 2: ai2}

            while True:
                self.render_board()

                current_name = self.player_names[self.game.current_player]
                console.print(f"[dim]{current_name} está a jogar...[/dim]")

                # -----------------------------
                # ESCOLHER JOGADA
                # -----------------------------
                move = self.get_move()   # PRIMEIRO obter jogada

                # depois validar
                if move is None:
                    console.print("[red]Erro ao obter jogada[/red]")
                    continue

                valid_moves = self.game.get_valid_moves()

                if move not in valid_moves:
                    console.print("[yellow]Jogada inválida ... fallback[/yellow]")
                    move = random.choice(valid_moves)
               #
                console.print(f"[bold]Jogada escolhida: {move}[/bold]")

                move_type, col = move
                self.game.make_move(move_type, col)
                moves_count += 1 # CONTADOR DE JOGADAS (OPCIONAL)

                # -----------------------------
                # RESULTADOS
                # -----------------------------
                winner = self.game.check_winner(last_move_type=move_type)

                if winner:
                    self.render_board()
                    winner_name = self.player_names[winner]
                    symbol = "X" if winner == 1 else "O"
                    console.print(f"[bold green]{winner_name} ({symbol}) venceu! 🏆[/bold green]")
                    console.print(f"[cyan]Total de jogadas: {moves_count}[/cyan]")

                    break

                draw = self.game.check_draw()

                if draw:
                    self.render_board()
                    console.print("[bold red]Empate[/bold red]")
                    console.print(f"[cyan]Total de jogadas: {moves_count}[/cyan]")
                    break

        except GameExit:
            console.print("\n[bold yellow]Jogo terminado pelo utilizador.[/bold yellow]")

        except KeyboardInterrupt:
            console.print("\n[bold yellow]Interrompido (Ctrl+C).[/bold yellow]")

        finally:
            console.print("[dim]Obrigado por jogar PopOut![/dim]")

    # =========================================================
    # Random SIMPLES (TEMPORÁRIA)
    # =========================================================

    def random_ai(self):
        return random.choice(self.game.get_valid_moves())


# =========================================================
# EXECUÇÃO
# =========================================================

if __name__ == "__main__":
    interface = GameInterface()
    interface.run()

A carregar modelo ID3...

 ID3 ativo!

Escolha o modo de jogo:

1 - Humano_A vs Humano_B

2 - Humano_A vs IA

...

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃ 0 ┃ 1 ┃ 2 ┃ 3 ┃ 4 ┃ 5 ┃ 6 ┃
┡━━━╇━━━╇━━━╇━━━╇━━━╇━━━╇━━━┩
│ X │ · │ · │ · │ · │ · │ · │
│ O │ · │ O │ · │ · │ · │ · │
│ X │ · │ X │ X │ · │ · │ · │
│ X │ · │ X │ X │ · │ · │ · │
│ O │ · │ O │ O │ O │ X │ · │
│ O │ · │ X │ O │ O │ O │ O │
└───┴───┴───┴───┴───┴───┴───┘

Turno: ID3 (X)

MCTS (O) venceu! 🏆

Total de jogadas: 22

Obrigado por jogar PopOut!

#**Análise Comparativa de Desempenho**

Com o intuito de realizar uma avaliação quantitativa rigorosa dos algoritmos implementados, estabeleceu-se um protocolo experimental composto por b
 50 partidas. Os testes confrontaram o modelo ID3 contra três perfis distintos de oponentes: Monte Carlo Puro, Monte Carlo com Heurística e um Agente Aleatório.

In [ ]:
from rich.console import Console
from rich.table import Table
import time
import os
import random

#from game import PopOutGame
#from mcts_ai import MCTS, MCTS_Heuristic
#from id3_popout_dataset import load_model, predict



In [ ]:
console = Console()


# =========================================================
# ID3 AI
# =========================================================

class ID3_AI:
    def __init__(self):
        self.model = load_model()

    def board_to_dict(self, game):
        sample = {}

        rows, cols = game.board.shape

        for r in range(rows):
            for c in range(cols):
                sample[f"cell_{r}_{c}"] = game.board[r][c]

        sample["current_player"] = game.current_player

        return sample

    def get_move(self, game):

        sample = self.board_to_dict(game)

        prediction = predict(self.model, sample)

        move_type, col = prediction.split("_")
        col = int(col)

        move = (move_type, col)

        if move in game.get_valid_moves():
            return move

        return random.choice(game.get_valid_moves())


# =========================================================
# RANDOM AI
# =========================================================

class RandomAI:
    def get_move(self, game):
        return random.choice(game.get_valid_moves())


# =========================================================
# MCTS AI
# =========================================================

class MCTS_AI:
    def __init__(self, iterations=700):
        self.mcts = MCTS(iterations=iterations)

    def get_move(self, game):
        return self.mcts.get_best_move(game)


# =========================================================
# MCTS HEURISTIC AI
# =========================================================

class MCTS_Heuristic_AI:
    def __init__(self, iterations=700):
        self.mcts = MCTS_Heuristic(iterations=iterations)

    def get_move(self, game):
        return self.mcts.get_best_move(game)


# =========================================================
# GAME SIMULATION
# =========================================================

def play_game(ai1, ai2):

    game = PopOutGame()

    total_moves = 0

    while True:

        current_player = game.current_player

        if current_player == 1:
            move = ai1.get_move(game)
        else:
            move = ai2.get_move(game)

        # fallback
        if move not in game.get_valid_moves():
            move = random.choice(game.get_valid_moves())

        move_type, col = move

        game.make_move(move_type, col)

        total_moves += 1

        winner = game.check_winner(last_move_type=move_type)

        if winner:
            return winner, total_moves

        if game.check_draw():
            return 0, total_moves


# =========================================================
# MATCH SERIES
# =========================================================

def evaluate(ai_name, ai, games=50):

    console.print(f"\n[bold cyan]ID3 vs {ai_name}[/bold cyan]")

    id3 = ID3_AI()

    id3_wins = 0
    opponent_wins = 0
    draws = 0

    total_moves = 0

    start_time = time.time()

    for i in range(games):

        # alternar quem começa
        if i % 2 == 0:
            winner, moves = play_game(id3, ai)

            if winner == 1:
                id3_wins += 1
            elif winner == 2:
                opponent_wins += 1
            else:
                draws += 1

        else:
            winner, moves = play_game(ai, id3)

            if winner == 1:
                opponent_wins += 1
            elif winner == 2:
                id3_wins += 1
            else:
                draws += 1

        total_moves += moves

    total_time = time.time() - start_time

    # =====================================================
    # GUARDAR RELATÓRIO
    # =====================================================

    os.makedirs("../4_reports", exist_ok=True)

    report_path = f"../4_reports/id3_vs_{ai_name}.txt"

    with open(report_path, "w", encoding="utf-8") as f:

        f.write(f"RESULTADOS: ID3 vs {ai_name}\n")
        f.write("=" * 40 + "\n\n")

        f.write(f"Jogos: {games}\n")
        f.write(f"Vitórias ID3: {id3_wins}\n")
        f.write(f"Vitórias {ai_name}: {opponent_wins}\n")
        f.write(f"Empates: {draws}\n")

        winrate = (id3_wins / games) * 100

        f.write(f"Win Rate ID3: {winrate:.2f}%\n")
        f.write(f"Média de Jogadas: {total_moves / games:.2f}\n")
        f.write(f"Tempo Médio/Jogo: {total_time / games:.2f}s\n")

    # =====================================================
    # TABELA
    # =====================================================

    table = Table(title=f"RESULTADOS: ID3 vs {ai_name}")

    table.add_column("Métrica", style="cyan")
    table.add_column("Valor", style="green")

    table.add_row("Jogos", str(games))
    table.add_row("Vitórias ID3", str(id3_wins))
    table.add_row(f"Vitórias {ai_name}", str(opponent_wins))
    table.add_row("Empates", str(draws))

    winrate = (id3_wins / games) * 100

    table.add_row("Win Rate ID3", f"{winrate:.2f}%")

    table.add_row(
        "Média de Jogadas",
        f"{total_moves / games:.2f}"
    )

    table.add_row(
        "Tempo Médio/Jogo",
        f"{total_time / games:.2f}s"
    )

    console.print(table)


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    evaluate(
        "Random",
        RandomAI(),
        games=50
    )

    evaluate(
        "MCTS",
        MCTS_AI(iterations=700),
        games=50
    )

    evaluate(
        "MCTS_Heuristico",
        MCTS_Heuristic_AI(iterations=700),
        games=50
    )

ID3 vs Random

  RESULTADOS: ID3 vs Random  
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Métrica          ┃ Valor  ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Jogos            │ 50     │
│ Vitórias ID3     │ 48     │
│ Vitórias Random  │ 2      │
│ Empates          │ 0      │
│ Win Rate ID3     │ 96.00% │
│ Média de Jogadas │ 16.08  │
│ Tempo Médio/Jogo │ 0.00s  │
└──────────────────┴────────┘

ID3 vs MCTS

   RESULTADOS: ID3 vs MCTS   
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Métrica          ┃ Valor  ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Jogos            │ 50     │
│ Vitórias ID3     │ 5      │
│ Vitórias MCTS    │ 45     │
│ Empates          │ 0      │
│ Win Rate ID3     │ 10.00% │
│ Média de Jogadas │ 13.32  │
│ Tempo Médio/Jogo │ 20.63s │
└──────────────────┴────────┘

ID3 vs MCTS_Heuristico

 RESULTADOS: ID3 vs MCTS_Heuristico  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Métrica                  ┃ Valor  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Jogos                    │ 50     │
│ Vitórias ID3             │ 1      │
│ Vitórias MCTS_Heuristico │ 49     │
│ Empates                  │ 0      │
│ Win Rate ID3             │ 2.00%  │
│ Média de Jogadas         │ 16.52  │
│ Tempo Médio/Jogo         │ 26.10s │
└──────────────────────────┴────────┘

##**Análise dos Resultados**

**Id3 vs. Random (96%)**

A análise estatística revela que, embora o Id3 não seja um algoritmo convencionalmente aplicado a jogos de tabuleiro estratégicos, a sua performance foi substancialmente superior à de um agente aleatório, atingindo uma taxa de vitória de 96%. Este resultado demonstra que o modelo foi capaz de extrair padrões relevantes do jogo PopOut e desenvolver uma capacidade de generalização eficaz a partir do conjunto de dados gerado via MCTS.

**Id3 vs. Monte Carlo Puro (10%)**

Em contraste, ao enfrentar o algoritmo de Monte Carlo (MCTS), o ID3 apresentou um desempenho reduzido, com apenas 10% de vitórias. Este cenário era expectável: enquanto o ID3 baseia as suas decisões em regras estáticas aprendidas previamente, o MCTS realiza uma exploração dinâmica e estatística do espaço de estados em tempo real, permitindo uma tomada de decisão muito mais robusta em problemas de elevada complexidade combinatória.

**Id3 vs. Monte Carlo com Heurística(2%)**

Por fim, a performance do ID3 foi ainda mais limitada perante o Monte Carlo com Heurística, com apenas 2% de taxa de vitória. Este diferencial acentua-se pelo facto de o agente MCTS não só processar análises estatísticas rigorosas, mas também integrar conhecimento de domínio específico (heurísticas) sobre as regras do PopOut. Esta combinação maximiza a eficácia do oponente, evidenciando as limitações de um classificador puramente indutivo como o ID3 perante métodos de pesquisa informada.

#**Considerações Finais**


Este projeto permitiu comparar, na prática, como dois tipos diferentes de algoritmos lidam com um jogo de tabuleiro como o PopOut. Enquanto o ID3 tenta aprender regras fixas através dos dados, o Monte Carlo calcula a melhor jogada no momento através de simulações. A implementação mostrou que, embora o ID3 consiga aprender o básico e ganhar facilmente de um jogador aleatório, ele ainda sente dificuldades contra a capacidade de análise estatística do Monte Carlo

Notou-se também que o sucesso de um modelo indutivo neste domínio está intrinsecamente ligado à qualidade e quantidade da base de treino. Os resultados obtidos validam a viabilidade de utilizar algoritmos de procura para "ensinar" modelos mais leves, mas também sublinham que, para jogos de estratégia com elevada profundidade, a combinação de heurísticas específicas com o poder computacional do MCTS permanece a abordagem mais robusta. Este trabalho consolidou conceitos fundamentais de Inteligência Artificial, desde o tratamento de dados sintéticos até à análise comparativa de desempenho entre diferentes paradigmas algorítmicos.

#**Referências**

S. J. Russell and P. Norvig, “Monte Carlo Tree Search,” in Artificial Intelligence: A Modern Approach, 4th ed. Hoboken, NJ: Pearson, 2020, ch. 5, sec. 5.4.

Google. (2026). Gemini 3 Flash [Modelo de linguagem de grande escala]. https://gemini.google.com

OpenAI. (2026). ChatGPT (GPT-4o) [Modelo de linguagem de grande escala]. https://chat.openai.com

Anthropic. (2026). Claude 3.5 Sonnet [Modelo de linguagem de grande escala]. https://www.anthropic.com